# S4.dev — attempt-1 generations on the 30 dev-plots (Kaggle T4)

**RUNNER ONLY.** Clones, installs, calls scripts. No logic lives here —
`CLAUDE.md`: *"If logic lives in a notebook cell, it cannot enter the paper."*

⛔ **NOT A RESULT.** These generations are the substrate `w` and τ are fitted
on (`protocol.md` §S4 decisions 1, 2, 19). **No Critic, no Reflector** — the
Critic requires `w` and τ, and fitting them from a loop that used them would be
circular. Attempt 1 only.

| | |
|---|---|
| generator | `google/gemma-3-12b-it`, **single arm** (registered 2026-08-12) |
| grid | 30 plots × 2 axis levels × 2 prompt arms = **120 generations** |
| conditions | **A: free length** (run 2026-08-16) · **B: length-controlled** (this run) |
| runtime | ~25 min per condition |

### Before running — Kaggle settings

1. **Accelerator: GPU T4 ×2**, **Internet: ON**
2. **Add Input → Models** → `google/gemma-3` → variation `transformers / gemma-3-12b-it`
3. **Add Input → Datasets** → your `bn-clean`
4. If an earlier attempt crashed: **Run → Restart session** (not just the
   kernel — a dead kernel still holds its VRAM).

Every code cell is **self-contained**: it re-derives its paths and `cd`s to the
repo, so a kernel restart mid-notebook costs nothing but time.

## 1. Environment — GPU, repo, dependencies

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


In [ ]:
# Clone once; pull on re-run. Neither loses the results/ directory, which is
# where the generations land and is NOT regenerable on another host.
import os, subprocess
REPO = '/kaggle/working/repo'
if os.path.isdir(REPO + '/.git'):
    print(subprocess.run(['git','-C',REPO,'pull','--ff-only'],
                         capture_output=True, text=True).stdout)
else:
    subprocess.run(['git','clone','-q',
                    'https://github.com/alphapie77/BSc_Thesis.git', REPO], check=True)
os.chdir(REPO)
print('cwd:', os.getcwd())
print(subprocess.run(['git','log','--oneline','-1'],
                     capture_output=True, text=True).stdout)


In [ ]:
# `-U transformers` IS NOT OPTIONAL. Kaggle ships transformers 5.0.0, whose
# loader does not apply the 4-bit `quantization_config` -- the model then loads
# in fp16 (~24 GB) and dies on a 16 GB card. The pilot ran on 5.15.0
# (results/env_snapshot_s4_kaggle.json) and 4-bit worked there. Matching the
# pilot's environment also matters for its own sake: Coakley et al. (2022)
# measure >6 pp of variation from environment alone.
!pip -q install -U transformers accelerate bitsandbytes 2>&1 | tail -3
!pip -q install chromadb sentence-transformers pyyaml 2>&1 | tail -2
print('\nRESTART THE KERNEL NOW if transformers was upgraded in this cell')
print('(Run > Restart session), then continue from the preflight cell.')


## 2. Preflight — stop here if anything fails

All three must hold: **transformers ≥ 5.15.0**, **bitsandbytes available**, and
**both GPUs ~empty**. Cheaper to know now than 25 minutes into a run.

- transformers too old → the 4-bit config is dropped *silently* and fp16 loads.
- GPU not empty → a previous kernel still holds VRAM: **Run → Restart session**.

In [ ]:
import importlib, torch, transformers
from packaging.version import parse as V
print('transformers', transformers.__version__, '| torch', torch.__version__)
PILOT_TRANSFORMERS = '5.15.0'   # ref: results/env_snapshot_s4_kaggle.json
tf_ok = V(transformers.__version__) >= V(PILOT_TRANSFORMERS)
print(f'transformers >= {PILOT_TRANSFORMERS} (the pilot version):', tf_ok)
try:
    bnb = importlib.import_module('bitsandbytes')
    print('bitsandbytes', getattr(bnb, '__version__', '?'))
except Exception as e:
    print('bitsandbytes import FAILED:', e)
from transformers.utils import is_bitsandbytes_available
bnb_ok = is_bitsandbytes_available()
print('is_bitsandbytes_available:', bnb_ok)
gpu_ok = True
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f'GPU{i}: {free/2**30:5.1f} GiB free of {total/2**30:.1f}')
    gpu_ok &= free/2**30 > 10
print('\nREADY' if (tf_ok and bnb_ok and gpu_ok) else
      '\nNOT READY -- upgrade transformers and restart, and/or Run > Restart session')


## 3. Model mount and corpus

The weights are **mounted read-only** from Kaggle Models — nothing is downloaded
and nothing is written to `/kaggle/working` (the 2026-08-12 `ENOSPC`:
`snapshot_download(local_dir=…)` keeps a second copy and 24 GB does not fit in a
20 GB working dir).

In [ ]:
import glob, os
hits = sorted(glob.glob('/kaggle/input/gemma-3/transformers/*12b-it*/*') +
              glob.glob('/kaggle/input/models/google/gemma-3/transformers/*12b-it*/*'))
assert hits, 'Add Input -> Models -> google/gemma-3 -> transformers/gemma-3-12b-it'
# Exported to the ENVIRONMENT, not just a Python name: IPython expands $VAR only
# on the first line of a ! cell, and the continued form silently passed an empty
# path on 2026-08-15.
os.environ['GEMMA_PATH'] = hits[-1]
print('GEMMA_PATH =', hits[-1])
assert os.path.exists(hits[-1] + '/config.json'), 'mount has no config.json'


In [ ]:
# bn_clean.csv is gitignored (size + licence) and arrives as a Dataset.
# Kaggle has used both mount layouts; try each and fail loudly if neither.
import os, shutil, glob
os.chdir('/kaggle/working/repo'); os.makedirs('data/cleaned', exist_ok=True)
src = (glob.glob('/kaggle/input/datasets/*/bn-clean/bn_clean.csv') +
       glob.glob('/kaggle/input/bn-clean/bn_clean.csv') +
       glob.glob('/kaggle/input/**/bn_clean.csv', recursive=True))
assert src, 'attach the bn-clean dataset (Add Input -> Datasets)'
shutil.copy(src[0], 'data/cleaned/bn_clean.csv')
print('copied from', src[0], '|', os.path.getsize('data/cleaned/bn_clean.csv'), 'bytes')


## 4. Build the R1-only index

**Expect `886` rows and digest `85fc2d7d…`.** A different digest means different
rows went in and nothing downstream is comparable — stop if so.

The script *refuses* to run if an R2 or Gold-300 id reaches it (inviolable rules
4 and 5), checked twice by two mechanisms.

In [ ]:
import os; os.chdir('/kaggle/working/repo')
!python src/agents/build_index.py --config configs/s4_index.yaml


In [ ]:
import json, os; os.chdir('/kaggle/working/repo')
m = json.load(open('results/s4_index_manifest.json'))
print('rows  :', m.get('n_rows') or m.get('n_indexed') or m)
print('digest:', str(m.get('digest',''))[:16])


## 5. Dry run — read the rendered prompt before generating

Prints the full prompt for both levels and both prompt arms. **Read it.** Two of
the three bugs found on 2026-08-11 were caught exactly here, after the tests
were already green.

This runs the **length-controlled** config, so the last instruction before the
closing line should read *"এক-দুই বাক্যে লেখো, ২০ শব্দের মধ্যে।"* — identical at
both levels. If it differs between levels, stop: that would re-tie length to
level by hand.

In [ ]:
import os; os.chdir('/kaggle/working/repo')
!python src/agents/run_devplots.py --config configs/s4_devplots_lenctl.yaml --dry-run 2>&1 | tail -40


## 6. Condition A — free length (**already run 2026-08-16; skip**)

Skip if `results/s4_devplot_generations.jsonl` is committed. Kept as a
**condition, not a superseded draft**: it measures what the generator does when
nobody constrains length — it reads *specific* as *long* (level 1 at 38–40 mean
words against level 0's 6–13), inverting the corpus, where level 1 is *shorter*
(8.85 vs 13.12) yet richer. `kapur2026length` predicted exactly that for machine
text.

In [ ]:
import os; os.chdir('/kaggle/working/repo')
assert os.environ.get('GEMMA_PATH'), 'run the model-mount cell first'
# ONE line. No backslash continuation -- see the GEMMA_PATH cell.
!python src/agents/run_devplots.py --config configs/s4_devplots.yaml --model-path arm_a=$GEMMA_PATH


## 7. Condition B — length-controlled (**this run**)

One sentence added to the prompt, **identical at both levels**: at most 20 words.
20 is derived from region A, not chosen — level 0 averages 13.12 words, level 1
averages 8.85, corpus median is 8. Everything else is byte-identical.

**Why this run exists.** In condition A, length alone separated the levels at
AUC **0.9894** (bn) and **1.0000** (en), and **0** length-matched pairs existed
in either arm — in the en arm the ranges did not even overlap. So no
length-neutral claim about axis control could be made *at all*.

⚠️ `2601.01768` finds LLMs track their own output length poorly, so this clause
is expected to shift the distribution rather than enforce a bound. **If the AUC
stays ≥ 0.90 and the matched slice stays empty, the control FAILED** — a
pre-committed, reportable outcome, not a reason to retry.

Resumable: each generation is appended to the JSONL as it completes and a re-run
skips completed keys, so a dropped session costs only the remainder.

In [ ]:
import os; os.chdir('/kaggle/working/repo')
assert os.environ.get('GEMMA_PATH'), 'run the model-mount cell first'
!python src/agents/run_devplots.py --config configs/s4_devplots_lenctl.yaml --model-path arm_a=$GEMMA_PATH


## 8. Save the outputs — **before the session can die**

⚠️ Kaggle wipes the disk between sessions and these generations **cannot be
regenerated bit-for-bit elsewhere** — the JSONL *is* the reproducibility
artifact. On 2026-08-15 a complete 120-generation run was lost to exactly this.

Run this cell, then press **Save Version → Quick Save** (it does not re-run the
notebook; it makes `/kaggle/working` survive the session). Then download into
`E:\\Research\\Thesis\\thesis\\results\\`.

In [ ]:
import os, shutil; os.chdir('/kaggle/working/repo')
for f in ['s4_devplot_lenctl_generations.jsonl',
          's4_devplot_lenctl_generations.md',
          's4_devplot_lenctl_generations.json']:
    shutil.copy('results/' + f, '/kaggle/working/' + f)   # raises if missing
    print('saved', f)
!python src/common/env_snapshot.py
!cp results/env_snapshot.json /kaggle/working/env_snapshot_s4dev_lenctl_kaggle.json
!ls -lh /kaggle/working/


In [ ]:
# Read the verdict here rather than filing it. Direction-free by design: the
# 2026-08-16 diagnostic PASSED while the confound was total, because it asked
# whether level 1 came out SHORTER and level 1 came out longer.
import json, os; os.chdir('/kaggle/working/repo')
rep = json.load(open('results/s4_devplot_lenctl_generations.json'))['result']
print('n generations   :', rep['n_generations'])
print('mean words/cell :', {k: round(v['mean_words'], 1)
                            for k, v in rep['per_cell'].items()})
print('length-only AUC :', rep['length_only_auc'])
print('matched pairs   :', rep['matched_pairs'], '(of 30 per arm)')
print('VERDICT         :', rep['length_confound'])
